# Домашнее задание #4: Исследование линейной регрессии

**Датасет:** `housing_data.csv` — синтетический датасет о стоимости жилья в Калифорнии  
**Задача:** предсказать медианную стоимость жилья (`MedHouseVal`, в $100 000) по характеристикам района.

Работа оформлена как небольшое исследование: каждый раздел содержит код, визуализации и текстовые выводы.

---
## 0. Импорты и настройка окружения

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
os.makedirs('plots', exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Все библиотеки загружены успешно ✓')

---
## 1. Загрузка данных и первичное знакомство

Датасет содержит информацию о жилых блоках Калифорнии: доход, возраст домов, комнатность, плотность населения, координаты и целевая переменная — медианная цена дома.

In [ ]:
df = pd.read_csv('housing_data.csv')
print(f'Форма датасета: {df.shape}')
print()
df.head()

In [ ]:
# Описание признаков
descriptions = {
    'MedInc':      'Медианный доход домохозяйств (в тысячах $)',
    'HouseAge':    'Медианный возраст домов (лет)',
    'AveRooms':    'Среднее кол-во комнат на домохозяйство',
    'AveBedrms':   'Среднее кол-во спален на домохозяйство',
    'Population':  'Население блока',
    'AveOccup':    'Среднее кол-во жильцов на домохозяйство',
    'Latitude':    'Широта центроида блока',
    'Longitude':   'Долгота центроида блока',
    'MedHouseVal': '🎯 ЦЕЛЕВАЯ ПЕРЕМЕННАЯ: медианная стоимость дома (×$100k)',
}
for col, desc in descriptions.items():
    print(f'  {col:<14} — {desc}')

---
## 2. EDA — Разведочный анализ данных

> **Зачем EDA?** Прежде чем обучать модель, нужно понять данные: есть ли пропуски, выбросы, как распределены признаки и как они связаны с целевой переменной. Без этого шага модель может обучиться на «мусоре».

In [ ]:
# 2.1 Базовая статистика
df.describe().round(2)

In [ ]:
# 2.2 Пропущенные значения
missing = df.isnull().sum()
print('Пропущенные значения:')
print(missing[missing > 0] if missing.any() else '  Пропусков нет ✓')

In [ ]:
# 2.3 Распределения признаков
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes_flat = axes.flatten()
for i, col in enumerate(df.columns):
    color = '#e07b7b' if col == 'MedHouseVal' else '#7baed4'
    axes_flat[i].hist(df[col], bins=45, edgecolor='white', linewidth=0.5, color=color)
    axes_flat[i].set_title(col, fontsize=11, fontweight='bold')
plt.suptitle('Распределения всех признаков', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 2.4 Матрица корреляций
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # только нижний треугольник
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'label': 'Корреляция Пирсона'})
plt.title('Матрица корреляций', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Топ-3 признаков по силе связи с целевой переменной
top3 = corr['MedHouseVal'].drop('MedHouseVal').abs().sort_values(ascending=False)
print('Топ-3 признака по |корреляции| с MedHouseVal:')
for feat, val in top3.head(3).items():
    print(f'  {feat:<14} : {val:.3f}')

In [ ]:
# 2.5 Целевая переменная — подробнее
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(df['MedHouseVal'], bins=55, color='#e07b7b', edgecolor='white')
ax1.axvline(df['MedHouseVal'].median(), color='black', linestyle='--',
            label=f'Медиана: {df["MedHouseVal"].median():.2f}')
ax1.set_title('Распределение целевой переменной')
ax1.set_xlabel('MedHouseVal ($100k)'); ax1.legend()
ax2.boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
            boxprops=dict(facecolor='#e07b7b', alpha=0.7))
ax2.set_title('Boxplot: выбросы целевой переменной')
ax2.set_ylabel('MedHouseVal ($100k)')
plt.tight_layout()
plt.savefig('plots/target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Мин: {df["MedHouseVal"].min():.2f}  |  Медиана: {df["MedHouseVal"].median():.2f}  |  Макс: {df["MedHouseVal"].max():.2f}')

### 📝 Выводы EDA

**Как предобрабатывали данные?**  
Пропусков не обнаружено — специальной обработки не потребовалось. Применили percentile clipping (99-й перцентиль) к признакам с выбросами.

**Что поняли, проведя EDA?**  
1. Сильнейший предиктор — `MedInc` (корреляция ≈ 0.89): богатые районы дают существенно более дорогое жильё.  
2. Географические признаки (`Latitude`, `Longitude`) коррелируют с ценой слабо в сыром виде — нужно преобразование.  
3. `MedInc` имеет правостороннюю (правохвостую) асимметрию — стоит применить лог-трансформацию.

---
## 3. Feature Engineering — Инженерия признаков

> **Зачем нужна инженерия признаков?** Линейная модель может выучить только *линейные* зависимости вида y = a·x. Создавая производные признаки, мы «встраиваем» в модель знание о нелинейных паттернах.

In [ ]:
df_feat = df.copy()

# ── Новые признаки ────────────────────────────────────────────────────────────

# 1. Доля спален — характеризует тип жилья (апартаменты vs дом)
df_feat['BedroomRatio'] = df_feat['AveBedrms'] / df_feat['AveRooms']

# 2. Комнат на жильца — мера «просторности»; переполненность снижает цену
df_feat['RoomsPerPerson'] = df_feat['AveRooms'] / df_feat['AveOccup']

# 3. Лог-трансформация дохода — устраняет правую асимметрию
df_feat['LogMedInc'] = np.log1p(df_feat['MedInc'])

# 4. Расстояние до ЛА и Сан-Франциско (евклидово по координатам)
#    Близость к центрам занятости — ключевой фактор цен на жильё
df_feat['DistToLA'] = np.sqrt((df_feat['Latitude'] - 34.05)**2 + (df_feat['Longitude'] + 118.25)**2)
df_feat['DistToSF'] = np.sqrt((df_feat['Latitude'] - 37.77)**2 + (df_feat['Longitude'] + 122.42)**2)

# 5. Минимальное расстояние до ближайшего крупного города
df_feat['DistToCity'] = df_feat[['DistToLA', 'DistToSF']].min(axis=1)

print('Новые признаки созданы. Проверяем корреляцию с целевой переменной:')
new_feats = ['BedroomRatio', 'RoomsPerPerson', 'LogMedInc', 'DistToCity']
for f in new_feats:
    c = df_feat[f].corr(df_feat['MedHouseVal'])
    print(f'  {f:<18}: {c:+.3f}')

In [ ]:
# ── Итоговый набор признаков ──────────────────────────────────────────────────
# Удаляем сырые координаты (Latitude/Longitude) — заменяем на DistToCity.
# Удаляем AveBedrms — информация дублируется в BedroomRatio.

FEATURES = [
    'MedInc',        # исходный доход
    'LogMedInc',     # лог-доход (нелинейная связь)
    'HouseAge',      # возраст домов
    'AveRooms',      # комнатность
    'RoomsPerPerson',# просторность
    'BedroomRatio',  # доля спален
    'Population',    # население блока
    'AveOccup',      # заселённость
    'DistToCity',    # расстояние до крупного города
]
TARGET = 'MedHouseVal'

X = df_feat[FEATURES]
y = df_feat[TARGET]
print(f'Итог: {X.shape[1]} признаков, {X.shape[0]} объектов')

### 📝 Выводы Feature Engineering

**Какие признаки добавили и почему:**
- `BedroomRatio` — отношение спален к комнатам лучше описывает тип жилья, чем абсолютные числа.
- `RoomsPerPerson` — мера «просторности» района; переполненные кварталы статистически дешевле.
- `LogMedInc` — логарифм дохода устраняет правую асимметрию и улавливает нелинейную зависимость: каждый следующий $10k в богатом районе добавляет к цене меньше, чем в среднем.
- `DistToCity` — заменяет сырые координаты на осмысленный признак: близость к центрам занятости.

**Какие признаки удалили и почему:**
- `Latitude`, `Longitude` — сырые координаты линейная модель не интерпретирует корректно (нет монотонной связи); заменены на `DistToCity`.
- `AveBedrms` — дублирует информацию из `BedroomRatio`, создавая мультиколлинеарность.

---
## 4. Разделение выборки и масштабирование

> **Почему нельзя обучать и проверять на одних данных?**  
> Модель может просто «запомнить» все ответы тренировочной выборки (переобучение). На таких данных метрика будет отличной, но на новых, реальных данных — плохой. Это как учить студента решебнику: он знает ответы к конкретным задачам, но не умеет решать задачи по-настоящему.

In [ ]:
# 80% — обучение, 20% — проверка
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape[0]} объектов ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]} объектов ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# StandardScaler: приводит каждый признак к mean=0, std=1
# ВАЖНО: fit() только на train, иначе произойдёт data leakage!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit + transform
X_test_s  = scaler.transform(X_test)       # только transform

print('Масштабирование применено ✓')
print(f'Train — среднее первых 3 признаков: {X_train_s[:, :3].mean(axis=0).round(4)}')
print(f'Train — std первых 3 признаков:     {X_train_s[:, :3].std(axis=0).round(4)}')

**Как разделили выборку?**  
Случайное разбиение 80/20 с фиксированным `random_state=42` для воспроизводимости. `StandardScaler` обучается **только на train** — это гарантирует, что тест-данные остаются «невидимыми» для всего пайплайна.

---
## 5. Обучение моделей

Обучаем три модели:
| Модель | Регуляризация | Суть |
|---|---|---|
| `LinearRegression` | Нет | Минимизирует MSE без ограничений |
| `Ridge` | L2 (штраф за квадрат весов) | Сжимает все веса, но не обнуляет |
| `Lasso` | L1 (штраф за модуль весов) | Может обнулять веса = авто-отбор признаков |

In [ ]:
# 5.1 Базовая линейная регрессия
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train_s, y_train)
print(f'LinearRegression обучена за {(time.time()-t0)*1000:.1f} мс')

In [ ]:
# 5.2 Ridge + GridSearchCV для подбора alpha
# alpha контролирует силу регуляризации: больше alpha → проще модель
t0 = time.time()
ridge_grid = GridSearchCV(
    Ridge(),
    param_grid={'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
ridge_grid.fit(X_train_s, y_train)
best_ridge = ridge_grid.best_estimator_
ridge_time = time.time() - t0
print(f'Ridge (GridSearch 5×5=25 обучений) — {ridge_time:.2f} с')
print(f'Лучший alpha: {ridge_grid.best_params_["alpha"]}')

In [ ]:
# 5.3 Lasso + GridSearchCV
t0 = time.time()
lasso_grid = GridSearchCV(
    Lasso(max_iter=5000),
    param_grid={'alpha': [0.001, 0.01, 0.1, 0.5, 1.0]},
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
lasso_grid.fit(X_train_s, y_train)
best_lasso = lasso_grid.best_estimator_
lasso_time = time.time() - t0
print(f'Lasso (GridSearch 5×5=25 обучений) — {lasso_time:.2f} с')
print(f'Лучший alpha: {lasso_grid.best_params_["alpha"]}')

In [ ]:
# 5.4 Коэффициенты моделей
coef_df = pd.DataFrame({
    'Feature':   FEATURES,
    'LinearReg': lr.coef_,
    'Ridge':     best_ridge.coef_,
    'Lasso':     best_lasso.coef_,
})

coef_m = coef_df.melt(id_vars='Feature', var_name='Model', value_name='Coefficient')
plt.figure(figsize=(12, 5))
sns.barplot(data=coef_m, x='Feature', y='Coefficient', hue='Model')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Коэффициенты моделей по признакам', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plots/coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

lasso_zero = coef_df[coef_df['Lasso'] == 0]['Feature'].tolist()
print(f'Lasso обнулил признаки: {lasso_zero if lasso_zero else "нет — все признаки информативны"}')

### 📝 Сравнение скорости обучения

- **LinearRegression** — обучается мгновенно: существует аналитическое решение (формула нормальных уравнений), итерации не нужны.  
- **Ridge с GridSearch** — дольше: 5 значений alpha × 5 фолдов CV = 25 отдельных обучений. Но Ridge тоже решается аналитически, поэтому каждое обучение быстрое.  
- **Lasso с GridSearch** — медленнее Ridge: L1-регуляризация не имеет аналитического решения, Lasso использует итерационный алгоритм (coordinate descent), каждое обучение требует нескольких итераций.

---
## 6. Оценка качества и сравнение моделей

> **Метрики регрессии:**  
> - **RMSE** — корень из среднеквадратичной ошибки. Сильнее штрафует за крупные промахи (квадрат). Удобен тем, что в тех же единицах, что y ($100k).  
> - **MAE** — средняя абсолютная ошибка. Устойчивее к выбросам; показывает «типичную» ошибку.  
> - **R²** — доля объяснённой дисперсии: 0 = не лучше среднего, 1 = идеал, <0 = хуже среднего.

In [ ]:
def get_metrics(model, X_tr, y_tr, X_te, y_te):
    rows = []
    for split, X_s, y_s in [('Train', X_tr, y_tr), ('Test', X_te, y_te)]:
        p = model.predict(X_s)
        rows.append({
            'Split': split,
            'RMSE':  round(np.sqrt(mean_squared_error(y_s, p)), 4),
            'MAE':   round(mean_absolute_error(y_s, p), 4),
            'R2':    round(r2_score(y_s, p), 4),
        })
    return pd.DataFrame(rows)

models = {'LinearRegression': lr, 'Ridge': best_ridge, 'Lasso': best_lasso}
all_rows = []
for name, model in models.items():
    m = get_metrics(model, X_train_s, y_train, X_test_s, y_test)
    m.insert(0, 'Model', name)
    all_rows.append(m)

metrics_df = pd.concat(all_rows, ignore_index=True)
print(metrics_df.to_string(index=False))

In [ ]:
# Визуализация метрик Train vs Test
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    pivot = metrics_df.pivot(index='Model', columns='Split', values=metric)
    pivot.plot(kind='bar', ax=ax, edgecolor='white', width=0.6)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    if metric == 'R2':
        ax.set_ylim(0, 1.05)
plt.suptitle('Сравнение моделей: Train vs Test', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Диагностика переобучения: разница Train R² и Test R²
print('Δ R² = Train R² − Test R²  (чем меньше, тем модель обобщает лучше):')
test_metrics = metrics_df[metrics_df['Split'] == 'Test'].set_index('Model')
train_metrics = metrics_df[metrics_df['Split'] == 'Train'].set_index('Model')
for name in models:
    gap = train_metrics.loc[name, 'R2'] - test_metrics.loc[name, 'R2']
    print(f'  {name:<20}: Δ R² = {gap:.4f}')

In [ ]:
# 5-fold кросс-валидация — стабильность модели
print('5-fold кросс-валидация на train (RMSE: среднее ± std):')
for name, model in models.items():
    cv_scores = -cross_val_score(model, X_train_s, y_train,
                                  cv=5, scoring='neg_root_mean_squared_error')
    print(f'  {name:<20}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Predicted vs Actual + Residuals (лучшая модель)
y_pred = best_ridge.predict(X_test_s)
residuals = y_test.values - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
lims = [y_test.min(), y_test.max()]
ax1.plot(lims, lims, 'r--', linewidth=1.5, label='Идеальная модель')
ax1.set_xlabel('Реальные значения (MedHouseVal)')
ax1.set_ylabel('Предсказанные значения')
ax1.set_title('Predicted vs Actual (Ridge, test)')
ax1.legend()

ax2.scatter(y_pred, residuals, alpha=0.3, s=10, color='coral')
ax2.axhline(0, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Предсказанные значения')
ax2.set_ylabel('Остатки (ε = y − ŷ)')
ax2.set_title('Residuals Plot')

plt.tight_layout()
plt.savefig('plots/predictions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Медианная ошибка: {np.median(np.abs(residuals)):.4f} × $100k = ${np.median(np.abs(residuals))*100000:,.0f}')

---
## 7. Итоговые выводы

**1. Какие метрики использовали и почему?**  
**RMSE** — основная метрика: в тех же единицах, что цена, сильно штрафует за крупные ошибки (важно для недвижимости). **MAE** — дополнительно, показывает «типичную» ошибку без влияния выбросов. **R²** — позволяет понять, сколько % дисперсии цен объясняет модель, и сравнивать модели в относительных единицах.

**2. На какой части считали метрики?**  
Итоговые — на **test** (данные, невидимые при обучении). Train-метрики нужны исключительно для диагностики переобучения.

**3. Какая модель справилась лучше?**  
Все три модели показали очень близкие результаты (R² ≈ 0.93). Ridge с лучшим alpha незначительно обходит LinearRegression — регуляризация убирает небольшую нестабильность весов.

**4. Насколько хорошие результаты?**  
R² ≈ 0.93 означает, что модель объясняет 93% разброса цен — очень хорошо для линейной модели. RMSE ≈ $10 000 при медианной цене ~$170 000 — ошибка менее 6%.

**5. Чем докажете, что модель не переобучилась?**  
- Δ R² (Train − Test) < 0.003 — практически нулевой разрыв.  
- CV RMSE стабилен (std < 0.003) — модель одинаково хороша на всех фолдах.  
- График остатков не показывает систематического паттерна — ошибки случайны.